<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/SMA_SlopeTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import yfinance as yf
print(yf.__version__)
import pandas as pd
import numpy as np
from datetime import datetime
import time
import random
import requests
from datetime import datetime, timedelta
from scipy.stats import linregress
# ensure reproducibility
random.seed(42)
print("Libraries Installed!")

0.2.66
Libraries Installed!


In [2]:
def rolling_regression_slope(series, window=10):
    """Rolling linear regression slope (price units per bar)."""
    def calc_slope(y):
        if len(y) < 2:
            return np.nan
        x = np.arange(len(y))
        slope, _, _, _, _ = linregress(x, y)
        return slope
    return series.rolling(window).apply(calc_slope, raw=False)



# Weekly Timeframe

In [50]:
df = yf.download('PG', period="10y", interval="1wk",auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
  df.columns = df.columns.get_level_values(0)  # keep only first level
df['10_week_SMA'] = df['Close'].rolling(window=10).mean()
df['30_week_SMA'] = df['Close'].rolling(window=30).mean()
df['slope_raw'] = rolling_regression_slope(df['30_week_SMA'], window=10)
df['slope_pct_per_week'] = df['slope_raw'] / df['30_week_SMA']          # fractional change per week
df['SMA_Slope'] = df['slope_pct_per_week'] * 52 * 100       # % per year

df.tail(10)

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume,10_week_SMA,30_week_SMA,slope_raw,slope_pct_per_week,SMA_Slope
Date,,,,,,,,,,
2025-11-03,146.979996,150.220001,144.460007,150.100006,47966300,152.110237,155.973423,-0.490458,-0.003145,-16.351402
2025-11-10,147.669998,149.380005,144.089996,146.380005,42109300,150.986360,155.320701,-0.513195,-0.003304,-17.181300
2025-11-17,150.919998,151.500000,145.009995,147.750000,48043800,150.398012,155.090303,-0.496825,-0.003203,-16.657981
2025-11-24,148.160004,150.410004,146.539993,150.100006,38270300,149.718373,154.751550,-0.465717,-0.003009,-15.649142
2025-12-01,143.449997,148.889999,142.509995,148.100006,57013800,148.919275,154.349826,-0.439684,-0.002849,-14.812810
2025-12-08,142.839996,143.110001,138.139999,142.910004,63600400,148.082016,153.742999,-0.437806,-0.002848,-14.807765
2025-12-15,144.460007,148.449997,143.199997,143.270004,65548200,147.662967,153.105350,-0.451308,-0.002948,-15.328022
2025-12-22,144.740005,145.639999,142.080002,143.710007,29243700,147.102106,152.344540,-0.479080,-0.003145,-16.352503
2025-12-29,141.789993,145.070007,141.240005,144.800003,27907800,146.137999,151.678384,-0.512518,-0.003379,-17.570698


# Daily Timeframe

In [51]:
df = yf.download('PG', period="2y", interval="1d",auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)  # keep only first level

df['5_day_SMA'] = df['Close'].rolling(window=5).mean()
df['50_day_SMA'] = df['Close'].rolling(window=50).mean()
# Slope for 5-day SMA (short-term trend)
df['slope5_raw'] = rolling_regression_slope(df['5_day_SMA'], window=10)
df['slope5_pct_per_day'] = df['slope5_raw'] / df['5_day_SMA']  # fractional change per day
df['slope5_annualized_pct'] = df['slope5_pct_per_day'] * 252 * 100  # % per year
# Slope for 50-day SMA (intermediate-term trend)
df['slope50_raw'] = rolling_regression_slope(df['50_day_SMA'], window=10)
df['slope50_pct_per_day'] = df['slope50_raw'] / df['50_day_SMA']  # fractional change per day
df['slope50_annualized_pct'] = df['slope50_pct_per_day'] * 252 * 100  # % per year

df.tail(10)

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume,5_day_SMA,50_day_SMA,slope5_raw,slope5_pct_per_day,slope5_annualized_pct,slope50_raw,slope50_pct_per_day,slope50_annualized_pct
Date,,,,,,,,,,,,,
2025-12-23,143.179993,143.729996,142.080002,142.490005,9541600,144.732001,146.965762,0.625515,0.004322,108.911560,-0.106954,-0.000728,-18.339266
2025-12-24,144.490005,144.740005,142.830002,142.899994,3259200,144.068002,146.893079,0.553176,0.003840,96.760111,-0.094714,-0.000645,-16.248563
2025-12-26,144.740005,145.639999,144.309998,144.309998,4711500,143.912003,146.859954,0.377685,0.002624,66.135301,-0.085969,-0.000585,-14.751617
2025-12-29,144.570007,145.070007,143.949997,144.800003,7662100,143.934003,146.780132,0.152691,0.001061,26.733184,-0.081348,-0.000554,-13.966188
2025-12-30,144.050003,144.460007,143.570007,144.289993,6006400,144.206003,146.654159,-0.011248,-0.000078,-1.965650,-0.081614,-0.000557,-14.023993
2025-12-31,143.309998,144.139999,143.229996,144.000000,5293200,144.232004,146.502264,-0.130521,-0.000905,-22.804437,-0.085561,-0.000584,-14.717435
2026-01-02,141.789993,143.339996,141.240005,143.110001,8946100,143.692001,146.326722,-0.190655,-0.001327,-33.436066,-0.092998,-0.000636,-16.015838
2026-01-05,140.369995,141.350006,139.600006,141.100006,12297300,142.817999,146.111261,-0.227018,-0.001590,-40.057027,-0.105468,-0.000722,-18.190185
2026-01-06,139.910004,141.419998,139.509995,140.389999,10421800,141.885999,145.886401,-0.262509,-0.001850,-46.623612,-0.123140,-0.000844,-21.270773


# 30 minutes

In [52]:
df = yf.download('PG', interval='30m', period='60d',auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
   df.columns = df.columns.get_level_values(0)  # keep only first level

df['65d_SMA'] = df['Close'].rolling(window=65).mean()
# Slope calculation: regression over recent 10 bars (~5 trading hours)
df['slope_raw'] = rolling_regression_slope(df['65d_SMA'], window=10)
# Normalize → fractional change per 30-minute bar
df['slope_norm'] = df['slope_raw'] / df['65d_SMA']
# Annualize to % per year (standard for 30m regular hours)
bars_per_year = 252 * 13
df['SMA_Slope'] = df['slope_norm'] * bars_per_year * 100

df.tail(10)

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume,65d_SMA,slope_raw,slope_norm,SMA_Slope
Datetime,,,,,,,,,
2026-01-07 16:00:00+00:00,138.925003,139.139999,138.750000,138.955002,538968,141.792236,-0.075034,-0.000529,-173.360281
2026-01-07 16:30:00+00:00,138.324997,138.970001,138.279999,138.925003,606191,141.709082,-0.076458,-0.000540,-176.752993
2026-01-07 17:00:00+00:00,138.139999,138.490005,138.119995,138.320007,562616,141.624005,-0.077829,-0.000550,-180.030562
2026-01-07 17:30:00+00:00,137.725006,138.169907,137.674393,138.149994,742478,141.530543,-0.079467,-0.000561,-183.942021
2026-01-07 18:00:00+00:00,138.565002,138.600006,137.619995,137.710007,998834,141.450466,-0.080749,-0.000571,-187.016066
2026-01-07 18:30:00+00:00,138.619995,138.740005,138.259995,138.559998,822690,141.368466,-0.081984,-0.000580,-189.985012
2026-01-07 19:00:00+00:00,138.160004,138.615005,138.050003,138.615005,740873,141.279728,-0.083071,-0.000588,-192.624432
2026-01-07 19:30:00+00:00,138.309998,138.469894,138.139999,138.169998,609699,141.193574,-0.083951,-0.000595,-194.785382
2026-01-07 20:00:00+00:00,138.585007,138.649994,138.199997,138.300003,668837,141.111036,-0.084859,-0.000601,-197.005522


In [53]:
df = yf.download('PG', interval='15m', period='30d',auto_adjust=True)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)  # keep only first level
df['130d_SMA'] = df['Close'].rolling(window=130).mean()

# Slope calculation: regression over recent 10 bars (~5 trading hours)
df['slope_raw'] = rolling_regression_slope(df['130d_SMA'], window=10)
# Normalize → fractional change per 15-minute bar
df['slope_norm'] = df['slope_raw'] / df['130d_SMA']
# Annualize to % per year (standard for 30m regular hours)
bars_per_year = 252 * 26
df['SMA_Slope'] = df['slope_norm'] * bars_per_year * 100
df.tail()



[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume,130d_SMA,slope_raw,slope_norm,SMA_Slope
Datetime,,,,,,,,,
2026-01-07 19:45:00+00:00,138.309998,138.419998,138.199997,138.350006,297827,141.188345,-0.042831,-0.000303,-198.760870
2026-01-07 20:00:00+00:00,138.429993,138.509995,138.199997,138.300003,339606,141.145306,-0.042767,-0.000303,-198.523822
2026-01-07 20:15:00+00:00,138.585007,138.649994,138.360001,138.410004,329231,141.104037,-0.042849,-0.000304,-198.962499
2026-01-07 20:30:00+00:00,138.220001,138.660004,138.154999,138.580002,422891,141.058921,-0.043053,-0.000305,-199.973940
2026-01-07 20:45:00+00:00,138.029999,138.271805,137.990005,138.225006,1492722,141.012460,-0.043271,-0.000307,-201.054849
